# Lab 3: Event Hub Streaming Consumer

This notebook consumes real-time weather events from Azure Event Hub and writes them into a Bronze Delta table using Spark Structured Streaming.

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("checkpoint_path", "")
dbutils.widgets.text("target_table", "weather_stream")

In [0]:
dbutils.widgets.text("eventhub_namespace", "")
dbutils.widgets.text("eventhub_name", "")

In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
target_table = dbutils.widgets.get("target_table")
eventhub_namespace = dbutils.widgets.get("eventhub_namespace")
eventhub_name = dbutils.widgets.get( "eventhub_name")

full_table_name = f"{catalog}.{bronze_schema}.{target_table}"

In [0]:
eventhub_connection_string = dbutils.secrets.get(
    scope="viktoriia_kalenichenko_scope",
    key="victoriya44250weatherconsumer"
)

In [0]:
# Kafka endpoint for the Event Hubs namespace
bootstrap_servers = f"{eventhub_namespace}.servicebus.windows.net:9093"


In [0]:
# authentication for Event Hubs via Kafka
sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{eventhub_connection_string}";'
)

In [0]:
df_raw = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", eventhub_name)
        .option("startingOffsets", "earliest")
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl_config)
        .load()
)

In [0]:
#schema showed value type is binary, so we need to fix this schema 
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

#for the future we can use this schema
weather_schema = StructType([
    StructField("event_timestamp", StringType(), True),
    StructField("country", StringType(), True),
    StructField("city", StringType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", IntegerType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True)
])

# Convert Kafka message body from binary to JSON string
df_string = df_raw.withColumn(
    "json_value",
    F.col("value").cast("string")
)

# Parse JSON into structured columns
df_parsed = (
    df_string
    .withColumn(
        "weather",
        F.from_json(F.col("json_value"), weather_schema)
    )
    .select(
        "weather.*",
        "partition",
        "offset",
        F.col("timestamp").alias("eventhub_timestamp")
    )
)

In [0]:
df_bronze = (
    df_parsed
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())

)



In [0]:
query = (
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(full_table_name)
)

In [0]:
query.awaitTermination()